
# Three NER Regimens **with** Joint Relation Extraction (SpERT‑style loss)

This notebook trains three NER setups while **jointly** learning Relation Extraction (RE) using the **same combined loss** as in `silver_to_gold_joint_ner_re.ipynb`:

1. **Silver-only** → (train on silver)  
2. **Silver → Gold** → (initialize from silver, then fine-tune on gold)  
3. **Gold-only** → (train on gold from the base encoder)

The **joint loss** is:  
\[ \mathcal{L} = \mathcal{L}_{\text{entity}} \; + \; \mathcal{L}_{\text{relation}} \]  
where both terms are cross-entropy losses over (a) **entity spans** and (b) **entity–entity pairs**.


In [1]:

# %% [setup]
import os, json, math, random, itertools, collections, glob
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ----------------
# Configuration
# ----------------
BASE_MODEL = "GroNLP/bert-base-dutch-cased"   # swap if you prefer another encoder
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Expected data layout (same as your previous notebooks):
# - Gold:   splits/train_val.json, splits/test.json
# - Silver: llm/bio/*.json  (all will be concatenated)
DATA_DIR   = Path(".")
OUT_SILVER = "BERT/joint_silver_phase"
OUT_GOLD   = "BERT/joint_gold_phase"
OUT_GOLD_ONLY = "BERT/joint_gold_only"

os.makedirs(OUT_SILVER, exist_ok=True)
os.makedirs(OUT_GOLD, exist_ok=True)
os.makedirs(OUT_GOLD_ONLY, exist_ok=True)

label_list = [
    "O",
    "B-Ontvanger", "I-Ontvanger",
    "B-Besluitvormend_orgaan", "I-Besluitvormend_orgaan",
    "B-Rechtsobject", "I-Rechtsobject",
    "B-Wettelijke_actie", "I-Wettelijke_actie",
    "B-Wettelijke_grondslag", "I-Wettelijke_grondslag",
]

print("Device:", DEVICE)


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:

# %% [helpers: IO + labels]
def load_json_list(path: Path) -> list:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a list of examples.")
    return data

def ensure_fields(ex, fields):
    for f in fields:
        if f not in ex:
            raise ValueError(f"Missing field {f!r} in example keys={list(ex.keys())}")

def build_old2new_from_examples(all_tag_lists: List[List[int]], label_list: Optional[List[str]]) -> Tuple[Dict[int,int], Dict[int,str], Dict[str,int]]:
    """Create a contiguous id mapping for actually-present label ids.
    If label_list is provided (BIO strings), id2label will use those; else we synthesize L{i}.
    """
    used = set()
    for tags in all_tag_lists:
        used.update(int(t) for t in tags)
    used = sorted(used)
    old2new = {old:i for i, old in enumerate(used)}
    if label_list is not None:
        id2label = {i: (label_list[old] if 0 <= old < len(label_list) else f"LABEL_{old}") for i, old in enumerate(used)}
    else:
        id2label = {i: f"L{i}" for i in range(len(used))}
    label2id = {v:k for k,v in id2label.items()}
    return old2new, id2label, label2id

def build_contig_to_base(old2new: Dict[int,int], label_list: Optional[List[str]]) -> Dict[int,str]:
    """Map contiguous ids -> *base* class (strip BIO prefix, keep 'O' as is).
    If no label_list is given, base becomes the id2label itself (L0, L1, ...).
    """
    contig_to_base = {}
    for old_id, new_id in old2new.items():
        if label_list is None:
            print('ha')
            name = f"L{new_id}" if old_id != 0 else "O"
        else:
            name = label_list[old_id] if 0 <= old_id < len(label_list) else "O"
        base = name.split("-", 1)[1] if "-" in name else name
        contig_to_base[new_id] = base
    return contig_to_base


In [3]:
label_list

['O',
 'B-Ontvanger',
 'I-Ontvanger',
 'B-Besluitvormend_orgaan',
 'I-Besluitvormend_orgaan',
 'B-Rechtsobject',
 'I-Rechtsobject',
 'B-Wettelijke_actie',
 'I-Wettelijke_actie',
 'B-Wettelijke_grondslag',
 'I-Wettelijke_grondslag']

In [4]:

# %% [data load]
gold_trainval_path = DATA_DIR / "splits" / "train_val.json"
gold_test_path     = DATA_DIR / "splits" / "test.json"
silver_glob        = str(DATA_DIR / "llm" / "bio" / "*.json")

gold_trainval = load_json_list(gold_trainval_path)
gold_test     = load_json_list(gold_test_path)
silver_files  = sorted(glob.glob(silver_glob))
silver_all    = [ex for p in silver_files for ex in load_json_list(Path(p))]


# Build contiguous label space across all sets that we might touch
all_tag_lists = [ex["ner_tags"] for ex in gold_trainval + gold_test + silver_all]
old2new, id2label_contig, label2id_contig = build_old2new_from_examples(all_tag_lists, label_list)
contig_to_base = build_contig_to_base(old2new, label_list)

# Base entity set (drop 'O')
BASE_LABELS = sorted(set(contig_to_base.values()) - {"O"})
ENT_LABELS  = ["O"] + BASE_LABELS
ent_label2id = {lab:i for i,lab in enumerate(ENT_LABELS)}
id2ent       = {i:lab for lab,i in ent_label2id.items()}

print("Entity base labels:", BASE_LABELS)
print("Total contiguous labels:", len(id2label_contig))
print("Silver files:", len(silver_files))
print(ent_label2id)


Entity base labels: ['Besluitvormend_orgaan', 'Ontvanger', 'Rechtsobject', 'Wettelijke_actie', 'Wettelijke_grondslag']
Total contiguous labels: 11
Silver files: 26
{'O': 0, 'Besluitvormend_orgaan': 1, 'Ontvanger': 2, 'Rechtsobject': 3, 'Wettelijke_actie': 4, 'Wettelijke_grondslag': 5}


In [5]:

# %% [RELATIONS schema]
# Directional, typed relations. Adjust as needed to match your project.
RELATIONS = {
    ("Ontvanger",             "Rechtsobject"):            "receives",
    ("Wettelijke_actie",      "Rechtsobject"):            "acts_on",
    ("Wettelijke_grondslag",  "Besluitvormend_orgaan"):   "authorizes",
    ("Besluitvormend_orgaan", "Wettelijke_actie"):        "performs",
}
rel_label2id = {"no_relation": 0}
for r in ["receives", "acts_on", "authorizes", "performs"]:
    rel_label2id[r] = len(rel_label2id)
id2rel = {v:k for k,v in rel_label2id.items()}

def relation_of(head_base: str, tail_base: str) -> str:
    return RELATIONS.get((head_base, tail_base), "no_relation")

print("Relation labels:", id2rel)


Relation labels: {0: 'no_relation', 1: 'receives', 2: 'acts_on', 3: 'authorizes', 4: 'performs'}


In [6]:

# %% [span + pair utilities]
def word_to_subword_bounds(tokenizer, words, max_length=256):
    enc = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=max_length)
    word_ids = enc.word_ids(0)
    first, last = {}, {}
    for i, w in enumerate(word_ids):
        if w is None: continue
        if w not in first: first[w] = i
        last[w] = i
    return enc, first, last

def base_spans_from_tags(tokens, ner_tags):
    """Collapse contiguous tokens with same *base* label into spans: (start_tok, end_tok_exclusive, base)"""
    spans = []
    i, n = 0, len(tokens)
    while i < n:
        old_id = int(ner_tags[i])
        if old_id not in old2new: i += 1; continue
        new_id = old2new[old_id]
        base = contig_to_base[new_id]
        if base != "O":
            s = i; j = i+1
            while j < n:
                oidj = int(ner_tags[j])
                if oidj not in old2new: break
                if contig_to_base[old2new[oidj]] != base: break
                j += 1
            spans.append((s, j, base))  # [s, j)
            i = j
        else:
            i += 1
    return spans

def sample_spans_and_pairs(spans, max_span_width=24, neg_span_ratio=1, neg_pair_ratio=4):
    """
    Create (a) span candidates and labels (entity classification) and
    (b) directed entity–entity pairs and labels (relation classification).
    - Positives for entities: the gold spans (with their base labels).
    - Negatives for entities: randomly sampled non-entity spans.
    - Positives for relations: all ordered pairs whose (base_head, base_tail) match RELATIONS.
    - Negatives for relations: other ordered pairs, subsampled by neg_pair_ratio.
    """
    # Entity spans
    pos_spans = spans[:]  # (s,e,base)
    pos_labels = [ent_label2id[b] for (_,_,b) in pos_spans]

    # Enumerate all spans up to max width
    all_spans = []
    if spans:
        n_tokens = max(e for (_,e,_) in spans)
    else:
        n_tokens = 0
    for s in range(n_tokens):
        for e in range(s+1, min(n_tokens, s+max_span_width)+1):
            all_spans.append((s,e))

    gold_set = {(s,e) for (s,e,_) in pos_spans}
    candidates = [se for se in all_spans if se not in gold_set]
    random.shuffle(candidates)
    neg_k = neg_span_ratio * max(1, len(pos_spans))
    neg_spans = candidates[:neg_k]
    neg_labels = [ent_label2id["O"]] * len(neg_spans)

    # Relations over *positive* entity spans only
    pos_pairs, pos_pair_labels = [], []
    all_pairs = []
    for i in range(len(pos_spans)):
        for j in range(len(pos_spans)):
            if i == j: continue
            ai = pos_spans[i]; bi = pos_spans[j]
            all_pairs.append((i,j))
            rel = relation_of(ai[2], bi[2])
            if rel != "no_relation":
                pos_pairs.append((i,j)); pos_pair_labels.append(rel_label2id[rel])

    # Negative pairs: non-positives
    neg_candidates = [p for p in all_pairs if p not in set(pos_pairs)]
    random.shuffle(neg_candidates)
    max_negs = neg_pair_ratio * max(1, len(pos_pairs))
    neg_pairs = neg_candidates[:max_negs]
    neg_pair_labels = [rel_label2id["no_relation"]] * len(neg_pairs)

    # Merge
    span_list  = pos_spans + [(s,e,"O") for (s,e) in neg_spans]
    span_y     = pos_labels + neg_labels
    pair_list  = pos_pairs + neg_pairs
    pair_y     = pos_pair_labels + neg_pair_labels
    return span_list, span_y, pair_list, pair_y


In [7]:

# %% [Joint dataset + collate]
class JointDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_length=256, max_span_width=24, neg_span_ratio=1, neg_pair_ratio=4):
        self.examples = examples
        self.tok = tokenizer
        self.max_len = max_length
        self.max_span_width = max_span_width
        self.neg_span_ratio = neg_span_ratio
        self.neg_pair_ratio = neg_pair_ratio

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        words = ex["tokens"]; tags = ex["ner_tags"]
        enc, first, last = word_to_subword_bounds(self.tok, words, max_length=self.max_len)

        spans = base_spans_from_tags(words, tags)
        span_list, span_y, pair_list, pair_y = sample_spans_and_pairs(
            spans, max_span_width=self.max_span_width, neg_span_ratio=self.neg_span_ratio, neg_pair_ratio=self.neg_pair_ratio
        )

        # Convert word-index spans to subword indices
        span_sub_ix = []
        for (s,e,_) in span_list:
            si = first.get(s, None); ei = last.get(e-1, None)
            span_sub_ix.append((si, ei))

        # Pair indices refer to POSITIVE span indices positions (only), so we map into span_list
        # NOTE: pair_list indices are over pos_spans only (first len(pos_spans) items in span_list)
        pair_sub_ix = []
        for (i_a, i_b) in pair_list:
            (s_a,e_a,_) = span_list[i_a]
            (s_b,e_b,_) = span_list[i_b]
            si_a = first.get(s_a, None); ei_a = last.get(e_a-1, None)
            si_b = first.get(s_b, None); ei_b = last.get(e_b-1, None)
            pair_sub_ix.append(((si_a, ei_a), (si_b, ei_b)))

        item = {k: v.squeeze(0) for k,v in enc.items()}
        item["span_sub_ix"]   = span_sub_ix
        item["pair_sub_ix"]   = pair_sub_ix
        item["entity_labels"] = torch.tensor(span_y, dtype=torch.long)
        item["pair_labels"]   = torch.tensor(pair_y, dtype=torch.long) if len(pair_y)>0 else torch.tensor([], dtype=torch.long)
        return item

def collate_joint(batch):
    assert len(batch) == 1, "Use batch_size=1 (variable span counts)."
    b = batch[0]
    out = {}
    for k, v in b.items():
        if torch.is_tensor(v) and k in ("input_ids", "attention_mask", "token_type_ids"):
            out[k] = v.unsqueeze(0)  # restore batch dim
        else:
            out[k] = v
    return out


In [8]:

# %% [SpERT-like joint model]
class JointSpert(nn.Module):
    def __init__(self, encoder_name, num_entity_labels, num_rel_labels, max_span_width=24, width_emb_dim=16, dropout=0.2):
        super().__init__()
        self.enc = AutoModel.from_pretrained(encoder_name)
        h = self.enc.config.hidden_size
        self.width_embeddings = nn.Embedding(max_span_width, width_emb_dim)

        self.ent_mlp = nn.Sequential(
            nn.Linear(2*h + width_emb_dim, h),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h, num_entity_labels)
        )
        self.rel_mlp = nn.Sequential(
            nn.Linear(3*h, h),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h, num_rel_labels)
        )

    def _span_repr(self, hidden, span_sub_ix):
        reps = []
        H = hidden.size(-1)
        for (si, ei) in span_sub_ix:
            if si is None or ei is None:
                reps.append(torch.zeros(2*H + self.width_embeddings.embedding_dim, device=hidden.device))
                continue
            hs = hidden[0, si]
            he = hidden[0, ei]
            width = torch.tensor(min(max(ei - si + 1, 1), self.width_embeddings.num_embeddings), device=hidden.device) - 1
            wemb = self.width_embeddings(width)
            reps.append(torch.cat([hs, he, wemb], dim=-1))
        if len(reps) == 0:
            return torch.zeros(0, 2*H + self.width_embeddings.embedding_dim, device=hidden.device)
        return torch.stack(reps, dim=0)

    def forward(self, input_ids, attention_mask, token_type_ids=None,
                span_sub_ix=None, pair_sub_ix=None,
                entity_labels=None, pair_labels=None, return_hidden=False):
        out = self.enc(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        hidden = out.last_hidden_state  # [1, T, H]

        span_repr = self._span_repr(hidden, span_sub_ix or [])
        ent_logits = self.ent_mlp(span_repr) if span_repr.numel() else torch.zeros(0, self.ent_mlp[-1].out_features, device=hidden.device)

        H = hidden.size(-1)
        pair_repr = []
        for (a, b) in pair_sub_ix or []:
            ai = self._span_repr(hidden, [a])
            bi = self._span_repr(hidden, [b])
            if ai.numel()==0 or bi.numel()==0:
                pair_repr.append(torch.zeros(3*H, device=hidden.device))
            else:
                ai = ai[0][:2*H]
                bi = bi[0][:2*H]
                a_start = ai[:H]; b_start = bi[:H]
                pair_repr.append(torch.cat([a_start, b_start, a_start*b_start], dim=-1))
        pair_repr = torch.stack(pair_repr, dim=0) if len(pair_repr)>0 else torch.zeros(0, 3*H, device=hidden.device)
        rel_logits = self.rel_mlp(pair_repr) if pair_repr.numel() else torch.zeros(0, self.rel_mlp[-1].out_features, device=hidden.device)

        # ----- Joint loss (same structure as in silver_to_gold_joint_ner_re.ipynb) -----
        loss = None
        if entity_labels is not None and span_repr.numel():
            ce_ent = nn.CrossEntropyLoss()
            loss_ent = ce_ent(ent_logits, entity_labels)
            loss_rel = torch.tensor(0.0, device=ent_logits.device)
            if pair_labels is not None and len(pair_labels)>0 and pair_repr.numel()>0:
                ce_rel = nn.CrossEntropyLoss()
                loss_rel = ce_rel(rel_logits, pair_labels)
            loss = 1.0 * loss_ent + 1.0 * loss_rel

        outd = {"loss": loss, "ent_logits": ent_logits, "rel_logits": rel_logits}
        if return_hidden:
            outd["hidden"] = hidden
        return outd


In [9]:

# %% [tokenizers, datasets, loaders]
joint_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def split_train_val(examples, val_ratio=0.1, seed=SEED):
    n = len(examples)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    k = max(1, int(val_ratio * n))
    val_idx = set(idx[:k])
    train = [examples[i] for i in range(n) if i not in val_idx]
    val   = [examples[i] for i in range(n) if i in val_idx]
    return train, val

# Gold splits
gold_train, gold_dev = split_train_val(gold_trainval, val_ratio=0.1, seed=SEED)

# Silver splits (if available); dev uses gold_dev to keep selection consistent
silver_train, silver_dev = split_train_val(silver_all, val_ratio=0.1, seed=SEED) if len(silver_all)>0 else ([], [])

# Datasets
train_joint_silver = JointDataset(silver_train, joint_tokenizer, max_length=256, max_span_width=24, neg_span_ratio=1, neg_pair_ratio=4) if len(silver_train)>0 else None
gold_train_joint   = JointDataset(gold_train,   joint_tokenizer, max_length=256, max_span_width=24, neg_span_ratio=1, neg_pair_ratio=4)
gold_dev_joint     = JointDataset(gold_dev,     joint_tokenizer, max_length=256, max_span_width=24, neg_span_ratio=1, neg_pair_ratio=4)
gold_test_joint    = JointDataset(gold_test,    joint_tokenizer, max_length=256, max_span_width=24, neg_span_ratio=1, neg_pair_ratio=4)

def make_loader(ds, shuffle, bs=1):
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collate_joint)

train_joint_silver = JointDataset(silver_train, joint_tokenizer, max_length=256, max_span_width=24, neg_span_ratio=1, neg_pair_ratio=4) if silver_train else None
dev_joint_silver   = JointDataset(silver_dev,   joint_tokenizer, max_length=256, max_span_width=24, neg_span_ratio=1, neg_pair_ratio=4) if silver_dev else None
# test_joint_silver = JointDataset(silver_test, joint_tokenizer, max_length=256, ...)  # optional

train_loader_silver = make_loader(train_joint_silver, shuffle=True,  bs=1) if train_joint_silver else None
dev_loader_silver   = make_loader(dev_joint_silver,   shuffle=False, bs=1) if dev_joint_silver   else None
# test_loader_silver = make_loader(test_joint_silver, shuffle=False, bs=1)  # optional

# Gold loaders stay for gold-only / silver→gold phases
train_loader_gold = make_loader(gold_train_joint, shuffle=True, bs=1)
dev_loader_gold   = make_loader(gold_dev_joint,   shuffle=False, bs=1)
test_loader_gold  = make_loader(gold_test_joint,  shuffle=False, bs=1)

print("Dataloaders ready.")


Dataloaders ready.


In [10]:
ex = gold_train_joint[5]

# Assumes: ex is your example dict, and you have joint_tokenizer, ent_label2id, rel_label2id
import torch

id2ent = {v: k for k, v in ent_label2id.items()}
id2rel = {v: k for k, v in rel_label2id.items()}

def bounds_from_span(span):
    # Accept (start,end) or a list of token indices → return [start, end)
    if isinstance(span, (list, tuple)) and len(span) == 2 and all(isinstance(x, int) for x in span):
        s, e = span
    else:
        idxs = list(span)
        s, e = min(idxs), max(idxs) + 1
    if e <= s:
        e = s + 1
    return s, e

def clamp(s, e, L):
    return max(0, min(s, L-1)), max(1, min(e, L))

def span_text(ids, span_item, tok):
    s, e = bounds_from_span(span_item)
    s, e = clamp(s, e, len(ids))
    return tok.decode(ids[s:e], skip_special_tokens=True).strip(), (s, e)

def is_int_like(x):
    return isinstance(x, int) or (torch.is_tensor(x) and x.dim()==0)

def resolve_span(ref, spans_list):
    """If ref is an index → take spans_list[ref]; otherwise ref is already a span."""
    if is_int_like(ref):
        return spans_list[int(ref)]
    return ref  # already a span (e.g., (start,end) or list of indices)

# --- pull fields ---
ids = ex["input_ids"].tolist() if hasattr(ex["input_ids"], "tolist") else list(ex["input_ids"])
spans = ex["span_sub_ix"]
ent_y = ex["entity_labels"].tolist() if hasattr(ex["entity_labels"], "tolist") else list(ex["entity_labels"])
pairs = ex["pair_sub_ix"]
pair_y = ex["pair_labels"].tolist() if hasattr(ex["pair_labels"], "tolist") else list(ex["pair_labels"])

# --- relations ---
rows = []
for k, pair in enumerate(pairs):
    # pair could be a tuple/list of ints (indices) OR of span-tuples; could also be a tensor
    if torch.is_tensor(pair):
        pair = pair.tolist()
    h_ref, t_ref = pair
    h_span = resolve_span(h_ref, spans)
    t_span = resolve_span(t_ref, spans)

    rel = id2rel.get(int(pair_y[k]), str(int(pair_y[k])))
    h_txt, h_bounds = span_text(ids, h_span, joint_tokenizer)
    t_txt, t_bounds = span_text(ids, t_span, joint_tokenizer)

    h_idx_for_type = int(h_ref) if is_int_like(h_ref) else None
    t_idx_for_type = int(t_ref) if is_int_like(t_ref) else None
    # If pair entries are direct spans, we can’t index ent_y; fall back to 'UNK' type.
    h_type = id2ent.get(ent_y[h_idx_for_type], "UNK") if h_idx_for_type is not None else "UNK"
    t_type = id2ent.get(ent_y[t_idx_for_type], "UNK") if t_idx_for_type is not None else "UNK"

    rows.append((h_txt, h_type, rel, t_txt, t_type, h_bounds, t_bounds))

# Print positives first, then a few negatives
positives = [r for r in rows if r[2] != "no_relation"]
negatives = [r for r in rows if r[2] == "no_relation"]

print(f"Positives: {len(positives)}  Negatives: {len(negatives)}\n")

for r in positives[:30]:
    print(f'( {r[0]} : {r[1]} @ {r[5]} )  --{r[2]}-->  ( {r[3]} : {r[4]} @ {r[6]} )')
for r in negatives[:10]:
    print(f'( {r[0]} : {r[1]} @ {r[5]} )  --no_relation-->  ( {r[3]} : {r[4]} @ {r[6]} )')


Positives: 3  Negatives: 9

( Wij : UNK @ (1, 2) )  --performs-->  ( verstrekken : UNK @ (2, 3) )
( verstrekken : UNK @ (2, 3) )  --acts_on-->  ( subsidie : UNK @ (5, 6) )
( u : UNK @ (3, 4) )  --receives-->  ( subsidie : UNK @ (5, 6) )
( verstrekken : UNK @ (2, 3) )  --no_relation-->  ( u : UNK @ (3, 4) )
( u : UNK @ (3, 4) )  --no_relation-->  ( verstrekken : UNK @ (2, 3) )
( verstrekken : UNK @ (2, 3) )  --no_relation-->  ( Wij : UNK @ (1, 2) )
( u : UNK @ (3, 4) )  --no_relation-->  ( Wij : UNK @ (1, 2) )
( Wij : UNK @ (1, 2) )  --no_relation-->  ( subsidie : UNK @ (5, 6) )
( subsidie : UNK @ (5, 6) )  --no_relation-->  ( u : UNK @ (3, 4) )
( subsidie : UNK @ (5, 6) )  --no_relation-->  ( verstrekken : UNK @ (2, 3) )
( Wij : UNK @ (1, 2) )  --no_relation-->  ( u : UNK @ (3, 4) )
( subsidie : UNK @ (5, 6) )  --no_relation-->  ( Wij : UNK @ (1, 2) )


In [11]:

# %% [training + evaluation]
def prf(tp, fp, fn):
    p = tp/(tp+fp) if (tp+fp) else 0.0
    r = tp/(tp+fn) if (tp+fn) else 0.0
    f = 2*p*r/(p+r) if (p+r) else 0.0
    return p, r, f

def run_epoch(model, loader, opt=None, desc=""):
    train = opt is not None
    model.train(train)
    tot, n, skipped = 0.0, 0, 0
    for batch in loader:
        inputs = {k: v.to(DEVICE) for k,v in batch.items() if k in ("input_ids","attention_mask","token_type_ids")}
        out = model(**inputs,
                    span_sub_ix=batch["span_sub_ix"],
                    pair_sub_ix=batch["pair_sub_ix"],
                    entity_labels=batch["entity_labels"].to(DEVICE),
                    pair_labels=batch["pair_labels"].to(DEVICE))
        loss = out["loss"]
        if loss is None or (torch.is_tensor(loss) and not loss.requires_grad):
            skipped += 1
            continue
        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        tot += float(loss.item()); n += 1
    avg = tot / max(1, n)
    if desc: print(f"{desc}: {avg:.4f}  (skipped {skipped} batches)")
    return avg


In [12]:
import os, json, shutil, torch
from pathlib import Path
from transformers import AutoTokenizer

def export_joint_spert_to_hf(
    model,
    out_dir,
    base_model_name,
    ent_labels,          # list[str] in *training order*
    rel_label2id,        # dict[str,int] (or {})
    max_span_width=24,
    width_emb_dim=16,
    dropout=0.2,
    use_safetensors=False
):
    """
    Write a Hugging Face-style checkpoint directory:
      - config.json: enough to rebuild JointSpert
      - pytorch_model.bin (or model.safetensors): raw state_dict
      - tokenizer files (from base model)
    """
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    id2label = {i: lab for i, lab in enumerate(ent_labels)}
    label2id = {lab: i for i, lab in enumerate(ent_labels)}

    cfg = {
        "model_type": "joint_spert",                 # freeform tag; helps you recognize it
        "base_model_name_or_path": base_model_name,  # e.g., "bert-base-cased"
        "num_entity_labels": len(ent_labels),
        "num_rel_labels": int(len(rel_label2id) if rel_label2id else 0),
        "max_span_width": int(max_span_width),
        "width_emb_dim": int(width_emb_dim),
        "dropout": float(dropout),
        "id2label": {str(k): v for k, v in id2label.items()},
        "label2id": label2id,
    }

    # 1) config.json
    with open(out / "config.json", "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)

    # 2) weights
    if use_safetensors:
        try:
            from safetensors.torch import save_file
            # NOTE: save_file expects a *tensor name -> tensor* dict
            state = {k: v.cpu() for k, v in model.state_dict().items()}
            save_file(state, str(out / "model.safetensors"))
        except Exception as e:
            print(f"[export] safetensors failed ({e}); falling back to PyTorch .bin")
            torch.save(model.state_dict(), out / "pytorch_model.bin")
    else:
        torch.save(model.state_dict(), out / "pytorch_model.bin")

    # 3) tokenizer (so you can AutoTokenizer.from_pretrained(out_dir))
    AutoTokenizer.from_pretrained(base_model_name).save_pretrained(out)

In [13]:
# %% [train: PHASE 1 — Silver-only]
joint_model = JointSpert(
    BASE_MODEL,
    num_entity_labels=len(ENT_LABELS),
    num_rel_labels=len(rel_label2id),
    max_span_width=24, width_emb_dim=16, dropout=0.2
).to(DEVICE)
opt = torch.optim.AdamW(joint_model.parameters(), lr=3e-5, weight_decay=0.01)

if train_loader_silver is not None and dev_loader_silver is not None:
    EPOCHS_SILVER = 2
    best_dev = float("inf")
    for ep in range(1, EPOCHS_SILVER+1):
        tr = run_epoch(joint_model, train_loader_silver, opt,  desc=f"silver train ep{ep}")
        va = run_epoch(joint_model, dev_loader_silver,   None, desc=f"silver dev   ep{ep}")
        if va < best_dev:
            best_dev = va
            # Save in **HF format**
            export_joint_spert_to_hf(
                model=joint_model,
                out_dir=OUT_SILVER,
                base_model_name=BASE_MODEL,
                ent_labels=ENT_LABELS,
                rel_label2id=rel_label2id,
                max_span_width=24,
                width_emb_dim=16,
                dropout=0.2,
                use_safetensors=False,   # set True if you prefer safetensors
            )
            print(f"[silver] saved HF checkpoint -> {OUT_SILVER}")
    print("Silver phase done.")
else:
    print("No (train+dev) silver data; skipping silver-only phase.")

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


silver train ep1: 0.1535  (skipped 405 batches)
silver dev   ep1: 0.0759  (skipped 43 batches)
[silver] saved HF checkpoint -> BERT/joint_silver_phase
silver train ep2: 0.0808  (skipped 405 batches)
silver dev   ep2: 0.0416  (skipped 43 batches)
[silver] saved HF checkpoint -> BERT/joint_silver_phase
Silver phase done.


In [16]:
# --- Put this near your imports (above Phase 2) ---
import os, json, torch
from pathlib import Path

def load_joint_spert_from_pretrained(hf_dir: str):
    """
    Rebuild JointSpert from an HF-style folder (config.json + pytorch_model.bin/model.safetensors)
    and load weights strictly.
    """
    hf_dir = Path(hf_dir)
    cfg_path = hf_dir / "config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"No config.json in {hf_dir}")

    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    # Rebuild model with the same hyperparams used in training
    base = cfg.get("base_model_name_or_path", "bert-base-cased")
    model = JointSpert(
        base,
        num_entity_labels=int(cfg["num_entity_labels"]),
        num_rel_labels=int(cfg.get("num_rel_labels", 0)),
        max_span_width=int(cfg.get("max_span_width", 24)),
        width_emb_dim=int(cfg.get("width_emb_dim", 16)),
        dropout=float(cfg.get("dropout", 0.2)),
    )

    # Locate weights
    weights_path = hf_dir / "model.safetensors"
    use_safetensors = weights_path.exists()
    if not use_safetensors:
        weights_path = hf_dir / "pytorch_model.bin"
        if not weights_path.exists():
            raise FileNotFoundError(f"No weights found in {hf_dir} (model.safetensors / pytorch_model.bin)")

    # Load weights
    if use_safetensors:
        from safetensors.torch import load_file
        state = load_file(str(weights_path))
    else:
        state = torch.load(str(weights_path), map_location="cpu")

    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print("[load_joint_spert_from_pretrained] WARNING: key mismatch")
        if missing:   print("  MISSING   (first 20):", missing[:20])
        if unexpected:print("  UNEXPECTED(first 20):", unexpected[:20])
        # If you expect a perfect match, switch to strict=True after fixing names.

    # Attach labels if present
    id2type = None
    if "id2label" in cfg:
        # cfg.id2label is {"0":"PER", ...}; make a list ordered by id
        id2type = [v for k,v in sorted(((int(k),v) for k,v in cfg["id2label"].items()), key=lambda x: x[0])]
    if id2type and hasattr(model, "id2type"):
        model.id2type = id2type

    return model

In [17]:
# %% [train: PHASE 2 — Silver → Gold fine-tuning (early stopping) — HF format]

from pathlib import Path
import os, torch

# 1) Load Silver checkpoint (prefer HF folder; fallback to legacy model.pt; else init fresh)
silver_dir = Path(OUT_SILVER)
gold_dir   = Path(OUT_GOLD)
gold_dir.mkdir(parents=True, exist_ok=True)

def _has_hf_checkpoint(d):
    d = Path(d)
    return (d/"config.json").exists() and ((d/"pytorch_model.bin").exists() or (d/"model.safetensors").exists())

if _has_hf_checkpoint(silver_dir):
    # Preferred: load from HF-style folder we exported in Phase 1
    joint_model = load_joint_spert_from_pretrained(str(silver_dir)).to(DEVICE)
    print(f"[gold] loaded HF silver checkpoint from {silver_dir}")
elif (silver_dir/"model.pt").exists():
    # Legacy fallback: load raw state_dict, then export HF later
    print(f"[gold] loading legacy silver state_dict from {silver_dir/'model.pt'}")
    joint_model = JointSpert(
        BASE_MODEL,
        num_entity_labels=len(ENT_LABELS),
        num_rel_labels=len(rel_label2id),
        max_span_width=24, width_emb_dim=16, dropout=0.2
    ).to(DEVICE)
    state = torch.load(silver_dir/"model.pt", map_location=DEVICE)
    joint_model.load_state_dict(state, strict=True)
else:
    # Silver was skipped: initialize from base
    print("[gold] no silver checkpoint found, initializing fresh from base.")
    joint_model = JointSpert(
        BASE_MODEL,
        num_entity_labels=len(ENT_LABELS),
        num_rel_labels=len(rel_label2id),
        max_span_width=24, width_emb_dim=16, dropout=0.2
    ).to(DEVICE)

opt = torch.optim.AdamW(joint_model.parameters(), lr=3e-5, weight_decay=0.01)

# 2) Fine-tune on GOLD with early stopping; save *HF-style* on improvement
best_dev = float("inf"); patience = 5; bad = 0; epoch = 0
while True:
    epoch += 1
    tr = run_epoch(joint_model, train_loader_gold, opt,  desc=f"gold train ep{epoch}")
    va = run_epoch(joint_model, dev_loader_gold,   None, desc=f"gold dev   ep{epoch}")

    if va + 1e-6 < best_dev:
        best_dev = va
        bad = 0

        # Save HF checkpoint for the current best gold model
        export_joint_spert_to_hf(
            model=joint_model,
            out_dir=gold_dir,                # e.g., BERT/gold_only_hf
            base_model_name=BASE_MODEL,
            ent_labels=ENT_LABELS,
            rel_label2id=rel_label2id,
            max_span_width=24,
            width_emb_dim=16,
            dropout=0.2,
            use_safetensors=False           # set True if you prefer safetensors
        )
        print(f"[gold] ↑ improved (dev={va:.6f}); saved HF checkpoint -> {gold_dir}")

    else:
        bad += 1
        print(f"[gold] no improvement (bad={bad}/{patience}); best_dev={best_dev:.6f}")
        if bad >= patience or epoch >= 20:
            break

print("Gold fine-tuning phase done.")

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[gold] loaded HF silver checkpoint from BERT\joint_silver_phase
gold train ep1: 0.3505  (skipped 388 batches)
gold dev   ep1: 0.4675  (skipped 50 batches)
[gold] ↑ improved (dev=0.467507); saved HF checkpoint -> BERT\joint_gold_phase
gold train ep2: 0.1539  (skipped 388 batches)
gold dev   ep2: 0.4365  (skipped 50 batches)
[gold] ↑ improved (dev=0.436516); saved HF checkpoint -> BERT\joint_gold_phase
gold train ep3: 0.1243  (skipped 388 batches)
gold dev   ep3: 0.4101  (skipped 50 batches)
[gold] ↑ improved (dev=0.410057); saved HF checkpoint -> BERT\joint_gold_phase
gold train ep4: 0.0972  (skipped 388 batches)
gold dev   ep4: 0.4603  (skipped 50 batches)
[gold] no improvement (bad=1/5); best_dev=0.410057
gold train ep5: 0.0855  (skipped 388 batches)
gold dev   ep5: 0.4320  (skipped 50 batches)
[gold] no improvement (bad=2/5); best_dev=0.410057
gold train ep6: 0.0663  (skipped 388 batches)
gold dev   ep6: 0.4316  (skipped 50 batches)
[gold] no improvement (bad=3/5); best_dev=0.410057


In [20]:
# %% [train: PHASE 3 — Gold-only (from scratch) — HF format]

from pathlib import Path
import torch

gold_only_dir = Path(OUT_GOLD_ONLY)
gold_only_dir.mkdir(parents=True, exist_ok=True)

# fresh init from base
gold_only_model = JointSpert(
    BASE_MODEL,
    num_entity_labels=len(ENT_LABELS),
    num_rel_labels=len(rel_label2id),
    max_span_width=24,
    width_emb_dim=16,
    dropout=0.2
).to(DEVICE)

opt_go = torch.optim.AdamW(gold_only_model.parameters(), lr=3e-5, weight_decay=0.01)

best_dev = float("inf")
patience = 5
bad = 0
epoch = 0

while True:
    epoch += 1
    tr = run_epoch(gold_only_model, train_loader_gold, opt_go, desc=f"[gold-only] train ep{epoch}")
    va = run_epoch(gold_only_model, dev_loader_gold,   None,   desc=f"[gold-only] dev   ep{epoch}")

    if va + 1e-6 < best_dev:
        best_dev = va
        bad = 0

        # Save Hugging Face-style checkpoint
        export_joint_spert_to_hf(
            model=gold_only_model,
            out_dir=gold_only_dir,
            base_model_name=BASE_MODEL,
            ent_labels=ENT_LABELS,
            rel_label2id=rel_label2id,
            max_span_width=24,
            width_emb_dim=16,
            dropout=0.2,
            use_safetensors=False   # set True if you want safetensors
        )
        print(f"[gold-only] ↑ improved (dev={va:.6f}); saved HF checkpoint -> {gold_only_dir}")

    else:
        bad += 1
        print(f"[gold-only] no improvement (bad={bad}/{patience}); best_dev={best_dev:.6f}")
        if bad >= patience or epoch >= 20:
            break

print("Gold-only phase done.")

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[gold-only] train ep1: 0.5512  (skipped 388 batches)
[gold-only] dev   ep1: 0.3073  (skipped 50 batches)
[gold-only] ↑ improved (dev=0.307344); saved HF checkpoint -> BERT\joint_gold_only
[gold-only] train ep2: 0.1361  (skipped 388 batches)
[gold-only] dev   ep2: 0.3459  (skipped 50 batches)
[gold-only] no improvement (bad=1/5); best_dev=0.307344
[gold-only] train ep3: 0.0946  (skipped 388 batches)
[gold-only] dev   ep3: 0.5899  (skipped 50 batches)
[gold-only] no improvement (bad=2/5); best_dev=0.307344
[gold-only] train ep4: 0.0908  (skipped 388 batches)
[gold-only] dev   ep4: 0.3488  (skipped 50 batches)
[gold-only] no improvement (bad=3/5); best_dev=0.307344
[gold-only] train ep5: 0.0804  (skipped 388 batches)
[gold-only] dev   ep5: 0.4052  (skipped 50 batches)
[gold-only] no improvement (bad=4/5); best_dev=0.307344
[gold-only] train ep6: 0.0711  (skipped 388 batches)
[gold-only] dev   ep6: 0.4432  (skipped 50 batches)
[gold-only] no improvement (bad=5/5); best_dev=0.307344
Gold-on

In [21]:

@torch.no_grad()
def evaluate_entities(model, loader):
    tp = collections.Counter(); fp = collections.Counter(); fn = collections.Counter()
    for batch in loader:
        words_enc = {k: v.to(DEVICE) for k,v in batch.items()
                     if k in ("input_ids","attention_mask","token_type_ids")}
        ent_logits = model(**words_enc, span_sub_ix=batch["span_sub_ix"]).get("ent_logits")
        if ent_logits.numel()==0: 
            continue
        pred = ent_logits.argmax(-1).cpu().tolist()
        gold = batch["entity_labels"].tolist()
        for p,g in zip(pred,gold):
            if g != ent_label2id["O"] and p != ent_label2id["O"]:
                if p == g: tp[g] += 1
                else:      fp[p] += 1; fn[g] += 1
            elif g == ent_label2id["O"] and p != ent_label2id["O"]:
                fp[p] += 1
            elif g != ent_label2id["O"] and p == ent_label2id["O"]:
                fn[g] += 1

    print("Entity metrics:")
    print(f"{'Entity':20s} {'P':>7s} {'R':>7s} {'F1':>7s}")

    # per-class
    for eid, name in sorted(id2ent.items()):
        if name == "O": 
            continue
        p,r,f = prf(tp[eid], fp[eid], fn[eid])
        print(f"{name:20s} {p:7.3f} {r:7.3f} {f:7.3f}")

    # CORRECT micro (pos-only): sum once across positive classes
    pos_ids = [eid for eid,name in id2ent.items() if name != "O"]
    T   = sum(tp[i] for i in pos_ids)
    F_p = sum(fp[i] for i in pos_ids)
    F_n = sum(fn[i] for i in pos_ids)
    P,R,F = prf(T, F_p, F_n)
    print(f"Micro (pos-only): P={P:.3f} R={R:.3f} F1={F:.3f}")

@torch.no_grad()
def evaluate_relations(model, loader):
    tp = collections.Counter(); fp = collections.Counter(); fn = collections.Counter()
    for batch in loader:
        words_enc = {k: v.to(DEVICE) for k,v in batch.items() if k in ("input_ids","attention_mask","token_type_ids")}
        rel_logits = model(**words_enc, pair_sub_ix=batch["pair_sub_ix"]).get("rel_logits")
        if rel_logits.numel()==0: continue
        pred = rel_logits.argmax(-1).cpu().tolist()
        gold = batch["pair_labels"].tolist()
        for p,g in zip(pred,gold):
            if g != rel_label2id["no_relation"] and p != rel_label2id["no_relation"]:
                if p == g: tp[g] += 1
                else:      fp[p] += 1; fn[g] += 1
            elif g == rel_label2id["no_relation"] and p != rel_label2id["no_relation"]:
                fp[p] += 1
            elif g != rel_label2id["no_relation"] and p == rel_label2id["no_relation"]:
                fn[g] += 1
    print("Relation metrics:")
    print(f"{'Relation':20s} {'P':>7s} {'R':>7s} {'F1':>7s}")
    T=F_p=F_n=0
    for rid, name in sorted(id2rel.items()):
        if rid == rel_label2id["no_relation"]: continue
        p,r,f = prf(tp[rid], fp[rid], fn[rid])
        print(f"{name:20s} {p:7.3f} {r:7.3f} {f:7.3f}")
        T += tp[rid]; F_p += sum(fp.values()); F_n += sum(fn.values())
    P,R,F = prf(T, F_p, F_n)
    print(f"Micro (pos-only): P={P:.3f} R={R:.3f} F1={F:.3f}")

In [22]:

# %% [evaluation]
print("\n=== Evaluate Silver→Gold model on gold test ===")
joint_model.load_state_dict(torch.load(os.path.join(OUT_GOLD, "model.pt"), map_location=DEVICE))
evaluate_entities(joint_model, test_loader_gold)
evaluate_relations(joint_model, test_loader_gold)

print("\n=== Evaluate Gold-only model on gold test ===")
gold_only_model.load_state_dict(torch.load(os.path.join(OUT_GOLD_ONLY, "model.pt"), map_location=DEVICE))
evaluate_entities(gold_only_model, test_loader_gold)
evaluate_relations(gold_only_model, test_loader_gold)

if os.path.exists(os.path.join(OUT_SILVER, "model.pt")):
    print("\n=== Evaluate Silver-only model on gold test ===")
    tmp = JointSpert(BASE_MODEL, num_entity_labels=len(ENT_LABELS), num_rel_labels=len(rel_label2id)).to(DEVICE)
    tmp.load_state_dict(torch.load(os.path.join(OUT_SILVER, "model.pt"), map_location=DEVICE))
    evaluate_entities(tmp, test_loader_gold)
    evaluate_relations(tmp, test_loader_gold)



=== Evaluate Silver→Gold model on gold test ===
Entity metrics:
Entity                     P       R      F1
Besluitvormend_orgaan   1.000   0.970   0.985
Ontvanger              1.000   0.983   0.991
Rechtsobject           0.976   0.976   0.976
Wettelijke_actie       1.000   1.000   1.000
Wettelijke_grondslag   1.000   0.938   0.968
Micro (pos-only): P=0.994 R=0.978 F1=0.986
Relation metrics:
Relation                   P       R      F1
receives               0.983   1.000   0.992
acts_on                0.977   1.000   0.988
authorizes             1.000   0.964   0.982
performs               0.964   1.000   0.982
Micro (pos-only): P=0.918 R=0.982 F1=0.949

=== Evaluate Gold-only model on gold test ===
Entity metrics:
Entity                     P       R      F1
Besluitvormend_orgaan   0.890   0.970   0.929
Ontvanger              1.000   0.966   0.983
Rechtsobject           0.976   0.988   0.982
Wettelijke_actie       0.965   1.000   0.982
Wettelijke_grondslag   0.939   0.969   0.954
M

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Entity metrics:
Entity                     P       R      F1
Besluitvormend_orgaan   0.953   0.910   0.931
Ontvanger              0.800   0.339   0.476
Rechtsobject           0.959   0.866   0.910
Wettelijke_actie       0.975   0.939   0.957
Wettelijke_grondslag   1.000   0.375   0.545
Micro (pos-only): P=0.949 R=0.748 F1=0.837
Relation metrics:
Relation                   P       R      F1
receives               0.931   0.915   0.923
acts_on                0.987   0.881   0.931
authorizes             1.000   0.929   0.963
performs               0.980   0.889   0.932
Micro (pos-only): P=0.894 R=0.687 F1=0.777


In [38]:
@torch.no_grad()
def show_predicted_entities(model, loader, tokenizer, max_examples=5):
    shown = 0
    for batch in loader:
        # forward
        words_enc = {k: v.to(DEVICE) for k,v in batch.items()
                     if k in ("input_ids","attention_mask","token_type_ids")}
        ent_logits = model(**words_enc, span_sub_ix=batch["span_sub_ix"]).get("ent_logits")
        if ent_logits.numel() == 0:
            continue

        # data
        pred = ent_logits.argmax(-1).cpu().tolist()
        gold = batch["entity_labels"]
        gold = gold.tolist() if hasattr(gold, "tolist") else gold
        spans = batch["span_sub_ix"]
        spans = spans.tolist() if hasattr(spans, "tolist") else spans

        # ids/tokens (keep specials to preserve indices!)
        ids0 = batch["input_ids"][0]
        ids0 = ids0.tolist() if hasattr(ids0, "tolist") else ids0

        def decode_span(si, ei):
            "Decode using inclusive indices; handle None safely."
            if si is None or ei is None:
                return None  # truncated / unmapped
            # ei is inclusive → slice up to ei+1
            return tokenizer.decode(
                ids0[si:ei+1],
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            ).strip()

        full_text = tokenizer.decode(ids0, skip_special_tokens=True, clean_up_tokenization_spaces=True).strip()

        pred_entities = []
        gold_entities = []
        for (si, ei), p, g in zip(spans, pred, gold):
            # GOLD
            if id2ent[g] != "O":
                mention = decode_span(si, ei)
                label = id2ent[g]
                if mention is None or not mention:
                    gold_entities.append(("[TRUNCATED/EMPTY]", label))
                else:
                    gold_entities.append((mention, label))

            # PRED
            if id2ent[p] != "O":
                mention = decode_span(si, ei)
                label = id2ent[p]
                if mention is None or not mention:
                    pred_entities.append(("[TRUNCATED/EMPTY]", label))
                else:
                    pred_entities.append((mention, label))

        print("Text:", full_text)
        print("Predicted entities:", pred_entities)
        print("Gold entities:", gold_entities)
        print("-" * 60)

        shown += 1
        if shown >= max_examples:
            break

In [39]:
print("\n=== Predicted entities: Silver→Gold model ===")
joint_model.load_state_dict(torch.load(os.path.join(OUT_GOLD, "model.pt"), map_location=DEVICE))
show_predicted_entities(joint_model, test_loader_gold, joint_tokenizer)

print("\n=== Predicted entities: Gold-only model ===")
gold_only_model.load_state_dict(torch.load(os.path.join(OUT_GOLD_ONLY, "model.pt"), map_location=DEVICE))
show_predicted_entities(gold_only_model, test_loader_gold, joint_tokenizer)

if os.path.exists(os.path.join(OUT_SILVER, "model.pt")):
    print("\n=== Predicted entities: Silver-only model ===")
    tmp = JointSpert(BASE_MODEL, num_entity_labels=len(ENT_LABELS), num_rel_labels=len(rel_label2id)).to(DEVICE)
    tmp.load_state_dict(torch.load(os.path.join(OUT_SILVER, "model.pt"), map_location=DEVICE))
    show_predicted_entities(tmp, test_loader_gold, joint_tokenizer)



=== Predicted entities: Silver→Gold model ===
Text: Ingevolge artikel 48 van de Wtt 2018 is DNB bevoegd een bestuurlijke boete op te leggen ter zake van overtreding van artikel 10, eerste lid, van de Wtt ( oud ) juncto artikel 7, tweede lid, van de Rib Wtt ( oud ) alsmede artikel 15 van de Wtt 2018
Predicted entities: [('artikel 48 van de Wtt 2018', 'Wettelijke_grondslag'), ('DNB', 'Besluitvormend_orgaan')]
Gold entities: [('artikel 48 van de Wtt 2018', 'Wettelijke_grondslag'), ('DNB', 'Besluitvormend_orgaan')]
------------------------------------------------------------
Text: Ingevolge artikel 1 : 80 van de Wft is DNB bevoegd een bestuurlijke boete op te leggen ter zake van de overtreding van de voorschriften gesteld bij of krachtens artikel 1 : 128, tweede lid, van de Wft.
Predicted entities: [('artikel 1 : 80 van de Wft', 'Wettelijke_grondslag'), ('DNB', 'Besluitvormend_orgaan')]
Gold entities: [('artikel 1 : 80 van de Wft', 'Wettelijke_grondslag'), ('DNB', 'Besluitvormend_orgaan')

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Text: Ingevolge artikel 48 van de Wtt 2018 is DNB bevoegd een bestuurlijke boete op te leggen ter zake van overtreding van artikel 10, eerste lid, van de Wtt ( oud ) juncto artikel 7, tweede lid, van de Rib Wtt ( oud ) alsmede artikel 15 van de Wtt 2018
Predicted entities: [('DNB', 'Besluitvormend_orgaan')]
Gold entities: [('artikel 48 van de Wtt 2018', 'Wettelijke_grondslag'), ('DNB', 'Besluitvormend_orgaan')]
------------------------------------------------------------
Text: Ingevolge artikel 1 : 80 van de Wft is DNB bevoegd een bestuurlijke boete op te leggen ter zake van de overtreding van de voorschriften gesteld bij of krachtens artikel 1 : 128, tweede lid, van de Wft.
Predicted entities: [('artikel 1 : 80 van de Wft', 'Wettelijke_grondslag'), ('DNB', 'Besluitvormend_orgaan')]
Gold entities: [('artikel 1 : 80 van de Wft', 'Wettelijke_grondslag'), ('DNB', 'Besluitvormend_orgaan')]
------------------------------------------------------------
Text: Wij hebben besloten u een voorscho

In [ ]:
# === Write BIO JSON predictions aligned to gold tokens & BIO ids ===
# Produces: predictions/{model}_pred.json with fields:
#   id (doc_id from test set), tokens (original), ner_tags (gold), pred_tags (predicted)
#
# Requirements (already present in your notebook):
#   - gold_test (list of examples with keys: "doc_id" (or "id"), "tokens", "ner_tags")
#   - test_loader_gold (batch_size=1, shuffle=False) wrapping the same examples, in the same order
#   - joint_tokenizer (or tokenizer), DEVICE
#   - models & paths: joint_model, gold_only_model, OUT_GOLD, OUT_GOLD_ONLY, OUT_SILVER, BASE_MODEL, ENT_LABELS, rel_label2id, JointSpert
#   - utils: word_to_subword_bounds, id2ent, ent_label2id
#
# If your gold uses a different id field name (e.g. "id" instead of "doc_id"), the code falls back automatically.

import os, json, collections, torch
from typing import Dict, List, Tuple, Optional

# ---------- 1) Learn BIO id mapping (which int means B-<base> vs I-<base>) from the gold set ----------
def learn_bio_ids_from_gold(
    gold_examples: List[dict],
    old2new: Dict[int,int],
    contig_to_base: Dict[int,str],
    o_id: int = 0
) -> Dict[str, Dict[str, int]]:
    """
    Returns: base -> {"B": <int_id_for_B>, "I": <int_id_for_I>}
    We infer this by scanning gold BIO sequences and deducing which original ids
    behave like B vs I for each base. This keeps pred_tags in the SAME id space as gold.
    """
    # Count occurrences of (role=B/I) per base -> which raw id was used most
    B_counts: Dict[str, collections.Counter] = collections.defaultdict(collections.Counter)
    I_counts: Dict[str, collections.Counter] = collections.defaultdict(collections.Counter)

    def oldid_base(old_id: int) -> str:
        new_id = old2new.get(int(old_id), None)
        if new_id is None: return "O"
        return contig_to_base.get(new_id, "O")

    for ex in gold_examples:
        tags = ex["ner_tags"]
        prev_base = "O"
        for i, old_id in enumerate(tags):
            b = oldid_base(old_id)
            if b == "O":
                prev_base = "O"
                continue
            if prev_base != b:
                B_counts[b][old_id] += 1
            else:
                I_counts[b][old_id] += 1
            prev_base = b

    bio_ids = {}
    for base in set(list(B_counts.keys()) + list(I_counts.keys())):
        # pick the most frequent ids; fall back to any seen if needed
        b_id = B_counts[base].most_common(1)[0][0] if B_counts[base] else None
        i_id = I_counts[base].most_common(1)[0][0] if I_counts[base] else None
        # If we didn't observe an I for this base (e.g., only single-token mentions), re-use B as I
        if i_id is None: i_id = b_id
        # If we didn't observe a B either (unlikely), skip mapping that base
        if b_id is None: continue
        bio_ids[base] = {"B": int(b_id), "I": int(i_id)}

    # Ensure we keep O id explicit for reference (not used here, but handy)
    bio_ids["O"] = {"B": int(o_id), "I": int(o_id)}
    return bio_ids

# ---------- 2) Subword span → word span mapping ----------
def subword_span_to_word_span(
    si: Optional[int], ei: Optional[int],
    first: Dict[int,int], last: Dict[int,int]
) -> Optional[Tuple[int,int]]:
    """
    Map subword [si, ei] (inclusive) to word-span [s, e) (exclusive).
    Uses exact reverse lookup when possible; otherwise falls back to nearest inside bounds.
    Returns None if mapping fails.
    """
    if si is None or ei is None:
        return None
    # Exact reverses
    first_rev = {v:k for k,v in first.items()}
    last_rev  = {v:k for k,v in last.items()}
    s = first_rev.get(si, None)
    e_end = last_rev.get(ei, None)
    if s is not None and e_end is not None and e_end >= s:
        return (s, e_end+1)

    # Fallback: find the smallest word index whose first >= si, and largest whose last <= ei
    # (keeps the span inside the subword window)
    candidate_starts = [w for w, sidx in first.items() if sidx is not None and sidx >= si]
    candidate_ends   = [w for w, eidx in last.items()  if eidx is not None and eidx <= ei]
    if candidate_starts and candidate_ends:
        s2 = min(candidate_starts)
        e2_end = max(candidate_ends)
        if e2_end >= s2:
            return (s2, e2_end+1)
    return None

# ---------- 3) Assign word-level BIO tags from predicted spans ----------
def assign_pred_bio_for_example(
    words: List[str],
    span_sub_ix: List[Tuple[Optional[int], Optional[int]]],
    span_pred_labels: List[int],
    tokenizer,
    bio_ids_by_base: Dict[str, Dict[str,int]],
    id2ent: Dict[int,str]
) -> List[int]:
    """
    Returns word-level pred_tags (list of ints) aligned to words.
    Overlaps: prefer longer spans first; do not overwrite already-set tags.
    """
    enc, first, last = word_to_subword_bounds(tokenizer, words, max_length=256)

    # Build candidate word-spans with their base class, skip 'O' and unmapped
    cand = []
    for (si, ei), lab_id in zip(span_sub_ix, span_pred_labels):
        base = id2ent.get(int(lab_id), "O")
        if base == "O": 
            continue
        wspan = subword_span_to_word_span(si, ei, first, last)
        if wspan is None: 
            continue
        s, e = wspan
        if not (0 <= s < e <= len(words)): 
            continue
        cand.append((s, e, base))

    # Sort by length desc so longer spans win
    cand.sort(key=lambda t: (t[1]-t[0]), reverse=True)

    O_id = bio_ids_by_base["O"]["B"]
    pred = [O_id] * len(words)

    for s, e, base in cand:
        ids = bio_ids_by_base.get(base)
        if not ids:
            # If we don't know the gold BIO ids for this base, skip (keeps O)
            continue
        B_id, I_id = ids["B"], ids["I"]
        for i in range(s, e):
            if pred[i] != O_id: 
                continue
            pred[i] = B_id if i == s else I_id
    return pred

# ---------- 4) End-to-end writer per model ----------
@torch.no_grad()
def dump_predictions_json(
    model,
    loader,
    examples: List[dict],
    tokenizer,
    bio_ids_by_base: Dict[str, Dict[str,int]],
    out_path: str
):
    """
    Writes a list of dicts:
      {"id": <doc_id>, "tokens": [...], "ner_tags": [...], "pred_tags": [...]}
    """
    model.eval()
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out = []

    # Walk loader and examples in lockstep (batch_size=1, shuffle=False)
    for idx, batch in enumerate(loader):
        ex = examples[idx]
        words = ex["tokens"]
        gold_tags = ex["ner_tags"]
        doc_id = ex.get("doc_id", ex.get("id", str(idx)))

        # Forward to get span logits
        words_enc = {k: v.to(DEVICE) for k,v in batch.items()
                     if k in ("input_ids","attention_mask","token_type_ids")}
        ent_logits = model(**words_enc, span_sub_ix=batch["span_sub_ix"]).get("ent_logits")
        if ent_logits is None or ent_logits.numel() == 0:
            pred_tags = [bio_ids_by_base["O"]["B"]] * len(words)
        else:
            pred_ids = ent_logits.argmax(-1).cpu().tolist()
            spans = batch["span_sub_ix"]
            if hasattr(spans, "tolist"): 
                spans = spans.tolist()
            pred_tags = assign_pred_bio_for_example(
                words=words,
                span_sub_ix=spans,
                span_pred_labels=pred_ids,
                tokenizer=tokenizer,
                bio_ids_by_base=bio_ids_by_base,
                id2ent=id2ent
            )

        out.append({
            "id": doc_id,
            "tokens": words,       # exactly as in test set
            "ner_tags": gold_tags, # gold unchanged
            "pred_tags": pred_tags
        })

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"Wrote {len(out)} items to {out_path}")

# ---------- 5) Build BIO id map from gold once, then dump per model ----------
# We reuse the same mapping objects you already have in the notebook:
#   - old2new, contig_to_base (built from gold+silver with your helpers)
# If you don't have them in scope, rebuild them the same way you did earlier.

bio_ids_by_base = learn_bio_ids_from_gold(
    gold_examples=gold_test,         # same list used to build the dataset
    old2new=old2new,
    contig_to_base=contig_to_base,
    o_id=0                          
)

tok = joint_tokenizer  # or `tokenizer`, whichever you use

# 1) Silver→Gold (joint) model
joint_model.load_state_dict(torch.load(os.path.join(OUT_GOLD, "model.pt"), map_location=DEVICE))
dump_predictions_json(
    model=joint_model,
    loader=test_loader_gold,
    examples=gold_test,
    tokenizer=tok,
    bio_ids_by_base=bio_ids_by_base,
    out_path="predictions/silver_to_gold_pred.json"
)

# 2) Gold-only model
gold_only_model.load_state_dict(torch.load(os.path.join(OUT_GOLD_ONLY, "model.pt"), map_location=DEVICE))
dump_predictions_json(
    model=gold_only_model,
    loader=test_loader_gold,
    examples=gold_test,
    tokenizer=tok,
    bio_ids_by_base=bio_ids_by_base,
    out_path="predictions/gold_only_pred.json"
)

# 3) Silver-only model (if present)
if os.path.exists(os.path.join(OUT_SILVER, "model.pt")):
    tmp = JointSpert(BASE_MODEL, num_entity_labels=len(ENT_LABELS), num_rel_labels=len(rel_label2id)).to(DEVICE)
    tmp.load_state_dict(torch.load(os.path.join(OUT_SILVER, "model.pt"), map_location=DEVICE))
    dump_predictions_json(
        model=tmp,
        loader=test_loader_gold,
        examples=gold_test,
        tokenizer=tok,
        bio_ids_by_base=bio_ids_by_base,
        out_path="predictions/silver_only_pred.json"
    )
    del tmp


Wrote 184 items to predictions/silver_to_gold_pred.json
Wrote 184 items to predictions/gold_only_pred.json


Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wrote 184 items to predictions/silver_only_pred.json


In [37]:
def show_gold_entities(batch, tokenizer, id2ent):
    ids0 = batch["input_ids"][0].tolist()
    text = tokenizer.decode(ids0, skip_special_tokens=True)
    print("Full text:\n", text)
    print("-"*60)

    spans  = batch["span_sub_ix"]
    labels = batch["entity_labels"]

    for (si, ei), lab_id in zip(spans, labels):
        lab = id2ent[int(lab_id)]
        if lab == "O":  # skip negatives
            continue
        mention = tokenizer.decode(ids0[si:ei+1], skip_special_tokens=True,
                                   clean_up_tokenization_spaces=True).strip()
        print(f"Span {si}-{ei}: {mention!r} → {lab}")

# Example usage on the batch you just printed:
show_gold_entities(batch, joint_tokenizer, id2ent)

Full text:
 Ingevolge artikel 48 van de Wtt 2018 is DNB bevoegd een bestuurlijke boete op te leggen ter zake van overtreding van artikel 10, eerste lid, van de Wtt ( oud ) juncto artikel 7, tweede lid, van de Rib Wtt ( oud ) alsmede artikel 15 van de Wtt 2018
------------------------------------------------------------
Span 4-11: 'artikel 48 van de Wtt 2018' → Wettelijke_grondslag
Span 13-14: 'DNB' → Besluitvormend_orgaan


In [ ]:
@torch.no_grad()
def evaluate_entities_token_level(model, loader):
    """
    Token-level evaluation built from span predictions & gold.
    Robust to ragged lists, packed tensors, single-span (0-D label) cases, and empty spans.
    """
    bg_id = ent_label2id["O"]

    tp = collections.Counter()
    fp = collections.Counter()
    fn = collections.Counter()

    # --- Normalizers ----------------------------------------------------------
    def _to_pair_list(spans):
        """Return list of (s,e) with [start,end) convention."""
        import numpy as np
        if torch.is_tensor(spans):
            spans = spans.detach().cpu()
            if spans.ndim == 2 and spans.size(-1) == 2:
                return [(int(s), int(e)) for s, e in spans.tolist()]
            if spans.ndim == 1 and spans.numel() == 2:
                s, e = spans.tolist()
                return [(int(s), int(e))]
            if spans.numel() == 0:
                return []
            raise ValueError(f"Unexpected tensor shape for spans: {tuple(spans.shape)}")
        # python lists / tuples
        if spans is None:
            return []
        if isinstance(spans, (list, tuple)):
            if len(spans) == 0:
                return []
            first = spans[0]
            if isinstance(first, (list, tuple)) and len(first) == 2:
                return [(int(s), int(e)) for s, e in spans]
            # single span like [s,e]
            if isinstance(first, (int, np.integer)) and len(spans) == 2:
                return [(int(spans[0]), int(spans[1]))]
        raise ValueError(f"Unrecognized spans container: {type(spans)} -> {spans}")

    def _to_label_list(labels):
        """Return list[int] for labels; accepts 0-D tensors (single value)."""
        if torch.is_tensor(labels):
            if labels.ndim == 0:
                return [int(labels.item())]
            return [int(x) for x in labels.detach().cpu().view(-1).tolist()]
        if isinstance(labels, (list, tuple)):
            return [int(x) for x in labels]
        # single python scalar
        return [int(labels)]

    # --- Project spans to token labels ---------------------------------------
    def spans_to_token_labels(seq_len, attention_mask, spans, span_labels):
        spans = _to_pair_list(spans)
        span_labels = _to_label_list(span_labels)
        # guard: lengths may mismatch if upstream filtered spans; zip will truncate
        tok_lab   = [bg_id] * seq_len
        tok_cover = [0]     * seq_len  # longest-span-wins

        for (s, e), lab in zip(spans, span_labels):
            if lab == bg_id:
                continue
            s = max(0, int(s)); e = min(seq_len, int(e))
            if e <= s:
                continue
            span_len = e - s
            for t in range(s, e):
                if attention_mask[t] == 0:
                    continue
                if span_len > tok_cover[t]:
                    tok_cover[t] = span_len
                    tok_lab[t]   = int(lab)
        return tok_lab

    # -------------------------------------------------------------------------
    for batch in loader:
        words_enc = {k: v.to(DEVICE) for k, v in batch.items()
                     if k in ("input_ids","attention_mask","token_type_ids")}
        out = model(**words_enc, span_sub_ix=batch["span_sub_ix"])
        ent_logits = out.get("ent_logits")
        if ent_logits is None or ent_logits.numel() == 0:
            continue

        attention_mask = batch["attention_mask"]        # [B, T]
        span_sub_ix    = batch["span_sub_ix"]           # ragged or [N,2]
        gold_span_lab  = batch["entity_labels"]         # aligns with span_sub_ix
        pred_span_lab  = ent_logits.argmax(-1)          # aligns with span_sub_ix

        # ---- Case A: ragged per-item spans
        if isinstance(span_sub_ix, (list, tuple)) and len(span_sub_ix) > 0 and isinstance(span_sub_ix[0], (list, tuple, torch.Tensor)):
            for i in range(len(attention_mask)):
                att_i = attention_mask[i].detach().cpu().tolist()
                T     = len(att_i)

                # Some datasets store empty lists for items without spans; handle safely.
                spans_i = span_sub_ix[i] if i < len(span_sub_ix) else []
                gold_i  = gold_span_lab[i] if i < len(gold_span_lab) else []

                # pred per-item may be a tensor (0-D if exactly one span)
                # Normalize inside spans_to_token_labels
                pred_i  = pred_span_lab[i] if isinstance(pred_span_lab, (list, tuple)) else pred_span_lab

                gold_tok = spans_to_token_labels(T, att_i, spans_i, gold_i)
                pred_tok = spans_to_token_labels(T, att_i, spans_i, pred_i)

                for t in range(T):
                    if att_i[t] == 0:
                        continue
                    g = gold_tok[t]; p = pred_tok[t]
                    if g != bg_id and p != bg_id:
                        if p == g: tp[g] += 1
                        else:      fp[p] += 1; fn[g] += 1
                    elif g == bg_id and p != bg_id:
                        fp[p] += 1
                    elif g != bg_id and p == bg_id:
                        fn[g] += 1

        else:
            # ---- Case B: packed spans with mapping
            span_batch_ix = batch.get("span_batch_ix", None)
            if span_batch_ix is None:
                raise ValueError("Packed spans detected. Provide batch['span_batch_ix'] to map spans to items.")

            span_batch_ix = span_batch_ix.detach().cpu().tolist()
            pred_span_lab = pred_span_lab.detach().cpu().tolist()
            gold_span_lab = gold_span_lab.detach().cpu().tolist()

            B, T = attention_mask.shape
            spans_per_item = [[] for _ in range(B)]
            gold_per_item  = [[] for _ in range(B)]
            pred_per_item  = [[] for _ in range(B)]

            span_sub_ix_np = span_sub_ix.detach().cpu().tolist() if torch.is_tensor(span_sub_ix) else span_sub_ix
            for (s, e), g, p, bi in zip(span_sub_ix_np, gold_span_lab, pred_span_lab, span_batch_ix):
                spans_per_item[bi].append((s, e))
                gold_per_item[bi].append(g)
                pred_per_item[bi].append(p)

            for i in range(B):
                att_i = attention_mask[i].detach().cpu().tolist()
                Ti    = len(att_i)
                gold_tok = spans_to_token_labels(Ti, att_i, spans_per_item[i], gold_per_item[i])
                pred_tok = spans_to_token_labels(Ti, att_i, spans_per_item[i], pred_per_item[i])

                for t in range(Ti):
                    if att_i[t] == 0:
                        continue
                    g = gold_tok[t]; p = pred_tok[t]
                    if g != bg_id and p != bg_id:
                        if p == g: tp[g] += 1
                        else:      fp[p] += 1; fn[g] += 1
                    elif g == bg_id and p != bg_id:
                        fp[p] += 1
                    elif g != bg_id and p == bg_id:
                        fn[g] += 1

    # ---- Metrics printout ----------------------------------------------------
    def prf(tp_c, fp_c, fn_c):
        P = tp_c / (tp_c + fp_c) if (tp_c + fp_c) > 0 else 0.0
        R = tp_c / (tp_c + fn_c) if (tp_c + fn_c) > 0 else 0.0
        F = (2 * P * R / (P + R)) if (P + R) > 0 else 0.0
        return P, R, F

    print("Token-level Entity metrics:")
    print(f"{'Entity':20s} {'P':>7s} {'R':>7s} {'F1':>7s}")
    for eid, name in sorted(id2ent.items()):
        if name == "O":
            continue
        p, r, f = prf(tp[eid], fp[eid], fn[eid])
        print(f"{name:20s} {p:7.3f} {r:7.3f} {f:7.3f}")

    T_micro  = sum(tp[eid] for eid, nm in id2ent.items() if nm != "O")
    FP_micro = sum(fp[eid] for eid, nm in id2ent.items() if nm != "O")
    FN_micro = sum(fn[eid] for eid, nm in id2ent.items() if nm != "O")
    Pm = T_micro / (T_micro + FP_micro) if (T_micro + FP_micro) > 0 else 0.0
    Rm = T_micro / (T_micro + FN_micro) if (T_micro + FN_micro) > 0 else 0.0
    Fm = (2 * Pm * Rm / (Pm + Rm)) if (Pm + Rm) > 0 else 0.0
    print(f"Micro (pos-only): P={Pm:.3f} R={Rm:.3f} F1={Fm:.3f}")


In [ ]:

# %% [evaluation]
print("\n=== Evaluate Silver→Gold model on gold test ===")
joint_model.load_state_dict(torch.load(os.path.join(OUT_GOLD, "model.pt"), map_location=DEVICE))
evaluate_entities(joint_model, test_loader_gold)
evaluate_entities_token_level(joint_model, test_loader_gold)
evaluate_relations(joint_model, test_loader_gold)

print("\n=== Evaluate Gold-only model on gold test ===")
gold_only_model.load_state_dict(torch.load(os.path.join(OUT_GOLD_ONLY, "model.pt"), map_location=DEVICE))
evaluate_entities(gold_only_model, test_loader_gold)
evaluate_entities_token_level(gold_only_model, test_loader_gold)
evaluate_relations(gold_only_model, test_loader_gold)

if os.path.exists(os.path.join(OUT_SILVER, "model.pt")):
    print("\n=== Evaluate Silver-only model on gold test ===")
    tmp = JointSpert(BASE_MODEL, num_entity_labels=len(ENT_LABELS), num_rel_labels=len(rel_label2id)).to(DEVICE)
    tmp.load_state_dict(torch.load(os.path.join(OUT_SILVER, "model.pt"), map_location=DEVICE))
    evaluate_entities(tmp, test_loader_gold)
    evaluate_entities_token_level(tmp, test_loader_gold)
    evaluate_relations(tmp, test_loader_gold)



=== Evaluate Silver→Gold model on gold test ===


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Entity metrics:
Entity                     P       R      F1
Besluitvormend_orgaan   0.985   0.970   0.977
Ontvanger              0.967   0.983   0.975
Rechtsobject           0.988   0.976   0.982
Wettelijke_actie       0.988   1.000   0.994
Wettelijke_grondslag   0.968   0.938   0.952
Micro (pos-only): P=0.913 R=0.900 F1=0.906
Token-level Entity metrics:
Entity                     P       R      F1
Besluitvormend_orgaan   1.000   0.923   0.960
Ontvanger              1.000   0.945   0.972
Rechtsobject           0.333   1.000   0.500
Wettelijke_actie       1.000   1.000   1.000
Wettelijke_grondslag   1.000   0.788   0.881
Micro (pos-only): P=0.955 R=0.894 F1=0.924
Relation metrics:
Relation                   P       R      F1
receives               0.983   1.000   0.992
acts_on                0.977   1.000   0.988
authorizes             1.000   0.964   0.982
performs               0.964   1.000   0.982
Micro (pos-only): P=0.918 R=0.982 F1=0.949

=== Evaluate Gold-only model on gold test

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Entity metrics:
Entity                     P       R      F1
Besluitvormend_orgaan   1.000   0.910   0.953
Ontvanger              0.778   0.356   0.488
Rechtsobject           0.973   0.866   0.916
Wettelijke_actie       0.987   0.939   0.963
Wettelijke_grondslag   1.000   0.438   0.609
Micro (pos-only): P=0.844 R=0.385 F1=0.529
Token-level Entity metrics:
Entity                     P       R      F1
Besluitvormend_orgaan   1.000   0.856   0.922
Ontvanger              0.000   0.000   0.000
Rechtsobject           0.059   0.125   0.080
Wettelijke_actie       1.000   0.875   0.933
Wettelijke_grondslag   1.000   0.451   0.622
Micro (pos-only): P=0.902 R=0.391 F1=0.545
Relation metrics:
Relation                   P       R      F1
receives               0.915   0.915   0.915
acts_on                0.986   0.857   0.917
authorizes             1.000   0.929   0.963
performs               0.980   0.889   0.932
Micro (pos-only): P=0.877 R=0.667 F1=0.758


In [ ]:
# %% [inspection]  Show GOLD vs PRED token labels, with examples

import pandas as pd
from transformers import AutoTokenizer

# ---------- Shared helpers (same conventions as evaluator) ----------
def _to_pair_list(spans):
    """Return list of (s,e) with [start,end) convention."""
    import numpy as np, torch as _torch
    if _torch.is_tensor(spans):
        spans = spans.detach().cpu()
        if spans.ndim == 2 and spans.size(-1) == 2:
            return [(int(s), int(e)) for s, e in spans.tolist()]
        if spans.ndim == 1 and spans.numel() == 2:
            s, e = spans.tolist()
            return [(int(s), int(e))]
        if spans.numel() == 0:
            return []
        raise ValueError(f"Unexpected tensor shape for spans: {tuple(spans.shape)}")
    if spans is None:
        return []
    if isinstance(spans, (list, tuple)):
        if len(spans) == 0:
            return []
        first = spans[0]
        if isinstance(first, (list, tuple)) and len(first) == 2:
            return [(int(s), int(e)) for s, e in spans]
        if isinstance(first, (int, np.integer)) and len(spans) == 2:
            return [(int(spans[0]), int(spans[1]))]
    raise ValueError(f"Unrecognized spans container: {type(spans)} -> {spans}")

def _to_label_list(labels):
    """Return list[int] for labels; accepts 0-D tensors (single value)."""
    import torch as _torch
    if _torch.is_tensor(labels):
        if labels.ndim == 0:
            return [int(labels.item())]
        return [int(x) for x in labels.detach().cpu().view(-1).tolist()]
    if isinstance(labels, (list, tuple)):
        return [int(x) for x in labels]
    return [int(labels)]

def spans_to_token_labels(seq_len, attention_mask, spans, span_labels, bg_id):
    """Longest-span-wins projection onto tokens."""
    spans = _to_pair_list(spans)
    span_labels = _to_label_list(span_labels)
    tok_lab   = [bg_id] * seq_len
    tok_cover = [0]     * seq_len
    for (s, e), lab in zip(spans, span_labels):
        if lab == bg_id:
            continue
        s = max(0, int(s)); e = min(seq_len, int(e))
        if e <= s:
            continue
        span_len = e - s
        for t in range(s, e):
            if attention_mask[t] == 0:
                continue
            if span_len > tok_cover[t]:
                tok_cover[t] = span_len
                tok_lab[t]   = int(lab)
    return tok_lab

# ---------- Main inspection function ----------
@torch.no_grad()
def inspect_token_predictions(
    model,
    loader,
    tokenizer=None,
    max_examples=5,
    show_only_errors=True,
    max_tokens_per_example=512,
):
    """
    Prints tables of tokens with GOLD vs PRED token labels for a few examples.
    - model: your trained model (on DEVICE)
    - loader: your test loader (expects keys used below)
    - tokenizer: optional; uses TOKENIZER or loads from BASE_MODEL if None
    - show_only_errors: if True, only rows where gold != pred
    - max_examples: stop after showing this many examples with at least one row
    """
    model.eval()

    # Prepare tokenizer
    if tokenizer is None:
        if "TOKENIZER" in globals() and TOKENIZER is not None:
            tokenizer = TOKENIZER
        else:
            tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

    bg_id = ent_label2id["O"]
    shown = 0

    for batch_idx, batch in enumerate(loader):
        # Forward
        words_enc = {k: v.to(DEVICE) for k, v in batch.items()
                     if k in ("input_ids","attention_mask","token_type_ids")}
        out = model(**words_enc, span_sub_ix=batch["span_sub_ix"])
        ent_logits = out.get("ent_logits")
        if ent_logits is None or ent_logits.numel() == 0:
            continue

        # Tensors / containers
        input_ids     = batch["input_ids"]
        attention     = batch["attention_mask"]
        span_sub_ix   = batch["span_sub_ix"]
        gold_span_lab = batch["entity_labels"]
        pred_span_lab = ent_logits.argmax(-1)

        # Two batching modes: ragged per-item spans OR packed spans with span_batch_ix
        ragged_case = isinstance(span_sub_ix, (list, tuple)) and len(span_sub_ix) > 0 and isinstance(
            span_sub_ix[0], (list, tuple, torch.Tensor)
        )

        if ragged_case:
            B = input_ids.shape[0]
            for i in range(B):
                ids_i = input_ids[i].detach().cpu().tolist()
                att_i = attention[i].detach().cpu().tolist()
                T     = min(len(ids_i), max_tokens_per_example)
                # decode tokens (subword-level; keeps alignment)
                toks_i = tokenizer.convert_ids_to_tokens(ids_i[:T])

                spans_i = span_sub_ix[i] if i < len(span_sub_ix) else []
                gold_i  = gold_span_lab[i] if i < len(gold_span_lab) else []
                pred_i  = pred_span_lab[i] if isinstance(pred_span_lab, (list, tuple)) else pred_span_lab

                gold_tok = spans_to_token_labels(T, att_i[:T], spans_i, gold_i, bg_id)
                pred_tok = spans_to_token_labels(T, att_i[:T], spans_i, pred_i, bg_id)

                rows = []
                for t in range(T):
                    if att_i[t] == 0:  # skip padding
                        continue
                    g = gold_tok[t]; p = pred_tok[t]
                    if show_only_errors and g == p:
                        continue
                    rows.append({
                        "example": f"batch{batch_idx}_item{i}",
                        "tok_idx": t,
                        "token": toks_i[t],
                        "gold": id2ent[g],
                        "pred": id2ent[p],
                        "✓": " " if g != p else "✓",
                    })

                if rows:
                    df = pd.DataFrame(rows, columns=["example","tok_idx","token","gold","pred","✓"])
                    display(df)
                    shown += 1
                    if shown >= max_examples:
                        return
        else:
            # Packed spans need mapping span -> item
            if "span_batch_ix" not in batch:
                raise ValueError("Packed spans detected; please provide batch['span_batch_ix'] to map spans to items.")
            span_batch_ix = batch["span_batch_ix"].detach().cpu().tolist()
            pred_span_lab = pred_span_lab.detach().cpu().tolist()
            gold_span_lab = gold_span_lab.detach().cpu().tolist()

            B = input_ids.shape[0]
            # Group spans by item
            spans_per_item = [[] for _ in range(B)]
            gold_per_item  = [[] for _ in range(B)]
            pred_per_item  = [[] for _ in range(B)]
            span_pairs     = span_sub_ix.detach().cpu().tolist() if torch.is_tensor(span_sub_ix) else span_sub_ix
            for (s,e), g, p, bi in zip(span_pairs, gold_span_lab, pred_span_lab, span_batch_ix):
                spans_per_item[bi].append((s,e))
                gold_per_item[bi].append(g)
                pred_per_item[bi].append(p)

            for i in range(B):
                ids_i = input_ids[i].detach().cpu().tolist()
                att_i = attention[i].detach().cpu().tolist()
                T     = min(len(ids_i), max_tokens_per_example)
                toks_i = tokenizer.convert_ids_to_tokens(ids_i[:T])

                gold_tok = spans_to_token_labels(T, att_i[:T], spans_per_item[i], gold_per_item[i], bg_id)
                pred_tok = spans_to_token_labels(T, att_i[:T], spans_per_item[i], pred_per_item[i], bg_id)

                rows = []
                for t in range(T):
                    if att_i[t] == 0:
                        continue
                    g = gold_tok[t]; p = pred_tok[t]
                    if show_only_errors and g == p:
                        continue
                    rows.append({
                        "example": f"batch{batch_idx}_item{i}",
                        "tok_idx": t,
                        "token": toks_i[t],
                        "gold": id2ent[g],
                        "pred": id2ent[p],
                        "✓": " " if g != p else "✓",
                    })

                if rows:
                    df = pd.DataFrame(rows, columns=["example","tok_idx","token","gold","pred","✓"])
                    display(df)
                    shown += 1
                    if shown >= max_examples:
                        return

    if shown == 0:
        print("(No rows to show — either no errors under current settings, or empty logits.)")


# ---------- Example calls (run after loading weights like in your last cell) ----------

print("\n=== Inspect: Silver→Gold model (first errorful examples) ===")
joint_model.load_state_dict(torch.load(os.path.join(OUT_GOLD, "model.pt"), map_location=DEVICE))
inspect_token_predictions(joint_model, test_loader_gold, max_examples=3, show_only_errors=True)

print("\n=== Inspect: Gold-only model (first errorful examples) ===")
gold_only_model.load_state_dict(torch.load(os.path.join(OUT_GOLD_ONLY, "model.pt"), map_location=DEVICE))
inspect_token_predictions(gold_only_model, test_loader_gold, max_examples=3, show_only_errors=True)

if os.path.exists(os.path.join(OUT_SILVER, "model.pt")):
    print("\n=== Inspect: Silver-only model (first errorful examples) ===")
    tmp = JointSpert(BASE_MODEL, num_entity_labels=len(ENT_LABELS), num_rel_labels=len(rel_label2id)).to(DEVICE)
    tmp.load_state_dict(torch.load(os.path.join(OUT_SILVER, "model.pt"), map_location=DEVICE))
    inspect_token_predictions(tmp, test_loader_gold, max_examples=3, show_only_errors=True)



=== Inspect: Silver→Gold model (first errorful examples) ===


,example,tok_idx,token,gold,pred,✓
0,batch7_item0,3,last,Wettelijke_grondslag,Rechtsobject,
1,batch7_item0,4,##neming,Wettelijke_grondslag,Rechtsobject,
2,batch7_item0,5,subsidies,Wettelijke_grondslag,Rechtsobject,
3,batch7_item0,6,van,Wettelijke_grondslag,Rechtsobject,
4,batch7_item0,7,het,Wettelijke_grondslag,Rechtsobject,
5,batch7_item0,8,Besluit,Wettelijke_grondslag,Rechtsobject,
6,batch7_item0,9,begroting,Wettelijke_grondslag,Rechtsobject,
7,batch7_item0,10,en,Wettelijke_grondslag,Rechtsobject,
8,batch7_item0,11,verantwoording,Wettelijke_grondslag,Rechtsobject,
9,batch7_item0,12,provincies,Wettelijke_grondslag,Rechtsobject,


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,example,tok_idx,token,gold,pred,✓
0,batch118_item0,7,artikel,Wettelijke_grondslag,O,
1,batch118_item0,8,49,Wettelijke_grondslag,O,
2,batch118_item0,9,##a,Wettelijke_grondslag,O,
3,batch118_item0,10,",",Wettelijke_grondslag,O,
4,batch118_item0,11,vijfde,Wettelijke_grondslag,O,
5,batch118_item0,12,lid,Wettelijke_grondslag,O,
6,batch118_item0,13,",",Wettelijke_grondslag,O,
7,batch118_item0,14,M,Wettelijke_grondslag,O,


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


...

,example,tok_idx,token,gold,pred,✓
0,batch1_item0,4,artikel,Wettelijke_grondslag,O,
1,batch1_item0,5,48,Wettelijke_grondslag,O,
2,batch1_item0,6,van,Wettelijke_grondslag,O,
3,batch1_item0,7,de,Wettelijke_grondslag,O,
4,batch1_item0,8,W,Wettelijke_grondslag,O,
5,batch1_item0,9,##t,Wettelijke_grondslag,O,
6,batch1_item0,10,##t,Wettelijke_grondslag,O,


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,example,tok_idx,token,gold,pred,✓
0,batch6_item0,2,Sch,Ontvanger,O,
1,batch6_item0,3,##lum,Ontvanger,O,
2,batch6_item0,4,##berger,Ontvanger,O,
3,batch6_item0,5,Off,Ontvanger,O,
4,batch6_item0,6,##s,Ontvanger,O,
5,batch6_item0,7,##hor,Ontvanger,O,
6,batch6_item0,8,##e,Ontvanger,O,
7,batch6_item0,9,Service,Ontvanger,O,
8,batch6_item0,10,##s,Ontvanger,O,
9,batch6_item0,11,L,Ontvanger,O,


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,example,tok_idx,token,gold,pred,✓
0,batch7_item0,3,last,Wettelijke_grondslag,Rechtsobject,
1,batch7_item0,4,##neming,Wettelijke_grondslag,Rechtsobject,
2,batch7_item0,5,subsidies,Wettelijke_grondslag,Rechtsobject,
3,batch7_item0,6,van,Wettelijke_grondslag,Rechtsobject,
4,batch7_item0,7,het,Wettelijke_grondslag,Rechtsobject,
5,batch7_item0,8,Besluit,Wettelijke_grondslag,Rechtsobject,
6,batch7_item0,9,begroting,Wettelijke_grondslag,Rechtsobject,
7,batch7_item0,10,en,Wettelijke_grondslag,Rechtsobject,
8,batch7_item0,11,verantwoording,Wettelijke_grondslag,Rechtsobject,
9,batch7_item0,12,provincies,Wettelijke_grondslag,Rechtsobject,


In [ ]:
# === Notebook 2: EXPORT FIRST-5 PREDICTIONS TO /mnt/data/nb2_preds.jsonl ===
import json
from typing import List, Tuple, Dict

Span = Tuple[int, int, str]  # (start_idx, end_idx_exclusive, label)

# ---------- CONFIG: name your 3 span-based models here ----------
MODELS_NB2 = [
    "nb2_span_model_A",
    "nb2_span_model_B",
    "nb2_span_model_C",
]

# ---------- YOU MUST IMPLEMENT THIS depending on your code ----------
def predict_spans_for_texts(model_alias: str, tokens_list: List[List[str]]) -> List[List[Span]]:
    """
    Return predicted spans for each sentence in tokens_list.
    Implement this by calling your span-based predictor.
    Expected output: list of span lists, same length/order as tokens_list.
    Example spans: [(start, end, "ONTVANGER"), ...]
    """
    # TODO: replace the mock below with your real inference
    # For example, if you have something like:
    #   spans = my_model_inference(globals()[model_alias], tokens_list)
    #   return spans
    return [[] for _ in tokens_list]  # placeholder: no predictions

# ---------- Helpers ----------
def render_annotated(tokens: List[str], spans: List[Span]) -> str:
    spans = sorted(spans, key=lambda x: (x[0], -x[1]))
    out = []
    i = 0
    for (s, e, lab) in spans:
        if s > i:
            out.extend(tokens[i:s])
        out.append("[" + " ".join(tokens[s:e]) + "|" + lab + "]")
        i = e
    if i < len(tokens):
        out.extend(tokens[i:])
    text = " ".join(out)
    text = text.replace(" ,", ",").replace(" .", ".").replace(" )", ")").replace("( ", "(")
    return text

# ---------- Get first 5 test examples with gold ----------
def get_first5_tokens_and_gold_spans():
    """
    Adapt this to your dataset.
    Return:
      tokens_list: List[List[str]]
      gold_spans_list: List[List[Span]]
    """
    tokens_list = []
    gold_spans_list = []

    # EXAMPLE if your dataset has 'tokens' and 'entities' with start/end indices:
    for i in range(min(5, len(test_dataset))):
         item = test_dataset[i]
         toks = list(item["tokens"])
         gold_spans = [(ent["start"], ent["end"], ent["label"]) for ent in item["entities"]]
         tokens_list.append(toks)
         gold_spans_list.append(gold_spans)

    # If your gold is in BIO tags, convert them to spans here.
    # Replace this with your real logic:
    #raise NotImplementedError("Fill get_first5_tokens_and_gold_spans() with your dataset access.")

    return tokens_list, gold_spans_list

# ---------- Build file ----------
tokens_list, gold_spans_list = get_first5_tokens_and_gold_spans()
N = len(tokens_list)

gold_rendered = [render_annotated(toks, spans) for toks, spans in zip(tokens_list, gold_spans_list)]

per_model_rendered = {}
for alias in MODELS_NB2:
    pred_spans_list = predict_spans_for_texts(alias, tokens_list)
    rendered = [render_annotated(toks, spans) for toks, spans in zip(tokens_list, pred_spans_list)]
    per_model_rendered[alias] = rendered

records = []
for i in range(N):
    rec = {
        "index": i,
        "tokens": tokens_list[i],
        "gold_rendered": gold_rendered[i],
    }
    for alias in MODELS_NB2:
        rec[alias] = per_model_rendered[alias][i]
    records.append(rec)

out_path = "nb2_preds.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Wrote {len(records)} examples to {out_path}")


NameError: name 'test_dataset' is not defined

In [ ]:
rel_label2id

{'no_relation': 0, 'receives': 1, 'acts_on': 2, 'authorizes': 3, 'performs': 4}

In [ ]:
import os, json, shutil, torch
from pathlib import Path

# ----- inputs you used to build the model -----
BASE_MODEL = BASE_MODEL                 # e.g. "bert-base-cased"
ENT_LABELS = ENT_LABELS                 # list of entity type names in training order
rel_label2id = rel_label2id             # your relation label map (if used)
cfg = dict(
    base_model_name_or_path = BASE_MODEL,
    num_entity_labels = len(ENT_LABELS),
    num_rel_labels = len(rel_label2id),
    max_span_width = 24,
    width_emb_dim = 16,
    dropout = 0.2,
    id2label = {i: lab for i, lab in enumerate(ENT_LABELS)},
    label2id = {lab: i for i, lab in enumerate(ENT_LABELS)},
    model_type = "joint_spert"
)

OUT_GOLD_ONLY = Path(OUT_GOLD_ONLY)           # your existing folder with model.pt + tokenizer files
HF_DIR = OUT_GOLD_ONLY.with_name(OUT_GOLD_ONLY.name + "_hf")  # e.g. ".../silver_hf"
HF_DIR.mkdir(parents=True, exist_ok=True)

# 1) write a minimal config.json (so we can reconstruct the class)
with open(HF_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

# 2) place weights under the expected filename
# (PreTrainedModel expects "pytorch_model.bin" or "model.safetensors")
shutil.copyfile(OUT_GOLD_ONLY / "model.pt", HF_DIR / "pytorch_model.bin")

# 3) copy tokenizer files if they’re in OUT_SILVER (you already saved them there)
for fn in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "vocab.txt", "merges.txt"]:
    src = OUT_GOLD_ONLY / fn
    if src.exists():
        shutil.copyfile(src, HF_DIR / fn)

print(f"Converted to HF-style directory: {HF_DIR}")


Converted to HF-style directory: BERT\joint_gold_only_hf
